In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 1.1 Projectile Motion with Drag

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume I — Elementary Mechanics",
    number="1.1",
    title="Projectile Motion with Drag",
    blurb="How air resistance reshapes the parabola: linear vs. quadratic "
    "drag, terminal velocity, and why the optimal launch angle drops "
    "below 45°.",
    difficulty="introductory",
    estimate="45–75 min",
)

## Notebook overview

The drag-free projectile (a parabola, range maximised at exactly $45°$) is
the first trajectory every physics student computes. It is also a fiction:
the moment a real ball, shell, or raindrop moves through air, a velocity-dependent
drag force bends the parabola into something asymmetric, shortens
its reach, and quietly moves the best launch angle below $45°$. This notebook
is about *seeing* those changes by integrating the equations of motion
directly, rather than memorising a closed form that only exists in vacuum.

We will (1) implement the two-dimensional equations of motion with both a
**linear** (Stokes) and a **quadratic** (Newton) drag term, (2) integrate to
ground impact with an event-located landing, (3) confirm the drag-free limit
reproduces the analytic parabola and its $45°$ optimum, (4) sweep the launch
angle to compare the three regimes, (5) check the linear-drag case against its
exact closed form, (6) measure the quadratic terminal velocity, (7) reach
the physical payoff (the optimal angle dropping below $45°$ once drag is on)
and (8) map how the approach to terminal velocity depends on drag strength. A
worked animation of two flights racing (vacuum against drag, where the motion
is genuinely the point) punctuates Exercise 2.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the
> mechanics of resistive motion see Nolting, *Theoretical Physics 1*
> {cite}`nolting1`, and Taylor, *Classical Mechanics* {cite}`taylor`, whose
> Chapter 2 treats the linear- and quadratic-drag regimes in detail.

## Theory in brief

### Newton's second law with drag

A projectile of mass $m$ feels gravity and a drag force that opposes its
velocity $\mathbf v$. Two regimes bracket reality:

- **Linear (Stokes) drag**, $\mathbf F_\mathrm{drag} = -b\,\mathbf v$, valid at
  low Reynolds number (small, slow objects in viscous fluids);
- **Quadratic (Newton) drag**, $\mathbf F_\mathrm{drag} = -c\,|\mathbf v|\,
  \mathbf v$, valid at high Reynolds number (everyday balls and shells in air).

Dividing by $m$ removes the mass from the dynamics. With per-unit-mass
coefficients $k = b/m$ and $\kappa = c/m$, Newton's law for the state
$(x, v_x, y, v_y)$ (with $y$ vertical and $|\mathbf v| = \sqrt{v_x^2+v_y^2}$)
becomes

```{math}
:label: eq-eom
\ddot x = -k\,v_x - \kappa\,|\mathbf v|\,v_x, \qquad
\ddot y = -g - k\,v_y - \kappa\,|\mathbf v|\,v_y .
```

These are coupled and, for $\kappa \neq 0$, nonlinear (the speed
$|\mathbf v|$ couples the two components), so in general we integrate them
numerically. Three limits give us something exact to check against.

### Vacuum limit ($k=\kappa=0$)

Motion separates into uniform horizontal drift and uniform vertical
acceleration. Launched from the ground at speed $v_0$ and angle $\theta$, the
projectile lands at range

```{math}
:label: eq-range
R(\theta) = \frac{v_0^2 \sin 2\theta}{g},
```

which is maximal at $\theta = 45°$. This is our primary correctness check: any
integrator, run with the drag terms switched off, must reproduce this curve
and this optimum.

### Linear drag has a closed form

Because each component is linear and decoupled, the Stokes case integrates by
hand. With initial velocity $(v_{x0}, v_{y0})$ from the origin,

```{math}
:label: eq-linear
x(t) = \frac{v_{x0}}{k}\left(1 - e^{-kt}\right), \qquad
y(t) = \frac{1}{k}\left(v_{y0} + \frac{g}{k}\right)\left(1 - e^{-kt}\right)
       - \frac{g}{k}\,t .
```

The horizontal coordinate saturates at $v_{x0}/k$ (the projectile can only
drift so far), and the vertical motion approaches a constant **terminal
velocity** $-g/k$. We will check the numeric solution against these formulas.

### Quadratic drag and its terminal velocity

The Newton case has no elementary closed form for the full trajectory, but the
terminal velocity still follows from balancing gravity against drag,
$g = \kappa\,v_\mathrm{term}^2$, so an object dropped from rest approaches

```{math}
:label: eq-terminal
v_{y,\,\mathrm{term}} = -\sqrt{\frac{g}{\kappa}} .
```

Finally (the physics payoff of the notebook), because drag penalises the long,
slow, high-arc trajectories more than the flat ones, the range-maximising angle
falls **below** $45°$, and falls further as the launch speed rises.

### The launch geometry

Every flight in this notebook starts the same way: a projectile leaves the
origin at speed $v_0$ and angle $\theta$ above the horizontal, climbs along an
arc bent by gravity $g$ (and, once it is switched on, by drag), and returns to
the ground at the range $R$. The schematic fixes that geometry.

In [ ]:
# (solution hidden on the public site)


---
## Setup

In [ ]:
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import validate
from ecp.animate import show

rng = np.random.default_rng(0)

# Physical parameters (SI). Per-unit-mass drag coefficients, so mass drops out.
G = 9.81  # gravitational acceleration [m/s^2]
K_LIN = 0.5  # linear (Stokes) coefficient k = b/m   [1/s]
K_QUAD = 0.02  # quadratic (Newton) coefficient κ = c/m  [1/m]
V0 = 30.0  # default launch speed [m/s]

## Exercise 1 — Implement the equations of motion

The dynamics are the two coupled second-order equations {eq}`eq-eom`: each
acceleration is gravity (vertical only) minus a damping rate $k + \kappa|\mathbf
v|$ times the corresponding velocity component. Writing them as a first-order
system in the state $\mathbf s = (x, v_x, y, v_y)$ is what lets a standard ODE
solver integrate them, and carrying *both* drag coefficients in one function
lets us select the vacuum, linear, or quadratic regime just by which one is
non-zero.

Write the right-hand side `rhs(t, s, k_lin, k_quad, g)` that returns
$(\dot x, \ddot x, \dot y, \ddot y)$ for {eq}`eq-eom`.

1. Unpack the state and form the speed $|\mathbf v| = \sqrt{v_x^2+v_y^2}$
   (`numpy.hypot`) and the effective damping rate $k + \kappa|\mathbf v|$.
2. Return the derivative vector, remembering that only the vertical component
   carries the $-g$ gravity term.

In [ ]:
# (solution hidden on the public site)


### Validation 1 — the drag-free limit is free fall

With both coefficients zero, the acceleration must be exactly $(0, -g)$
regardless of velocity. Evaluate `rhs` at a random state and check.

In [ ]:
state = rng.uniform(-10.0, 10.0, size=4)
flow = rhs(0.0, state, k_lin=0.0, k_quad=0.0)
accel = [flow[1], flow[3]]
validate.close(accel, [0.0, -G], "drag-free limit is free fall", rtol=1e-12)

## Exercise 2 — Integrate to impact and plot a trajectory

With {eq}`eq-eom` in hand as a first-order system, an adaptive solver can carry
the state forward in time. Two numerical ideas make the result clean: a
**terminal event** stops the integration exactly at ground impact ($y$ crossing
zero on the way down) rather than at some arbitrary final time, and **dense
output** lets us resample the few stored steps onto a fine grid so the plotted
path is a smooth curve, not a polygon (see the note in `trajectory`).

Build the integrator and visualise the result.

1. Write `hit_ground` (the terminal event) and `fly(...)`, which launches from
   the origin at speed $v_0$ and angle $\theta$ and integrates with
   `scipy.integrate.solve_ivp` (`DOP853`, `dense_output=True`, the event
   attached).
2. Write `ground_range` (the located impact distance) and `trajectory` (the
   dense resampling), then overlay the vacuum and quadratic-drag paths for the
   same launch ($v_0 = 30$ m/s, $\theta = 45°$): drag visibly shortens both the
   reach and the height.
3. Animate the two projectiles flying side by side (the worked solution below,
   `FuncAnimation`), then confirm with a check that the drag flight lands
   shorter.

In [ ]:
# (solution hidden on the public site)


### A worked animation — the two flights, side by side

The static overlay shows the *paths*; an animation shows the *motion*. We sample
both flights on a common clock (each on its own dense output up to its own
landing time) and watch the drag projectile fall behind and land first. This is
the worked example for the two-animation rule; you build the second one in
Exercise 8.

In [ ]:
# (solution hidden on the public site)


### Validation 2 — drag shortens the range

Same launch, drag on: the quadratic flight must land closer than the vacuum
flight. This checks the *physics carried by the animation* (the relative reach),
not the drawing code.

In [ ]:
validate.check(
    ground_range(sol_quad) < ground_range(sol_vac),
    "drag shortens the range at 45°",
    f"R_quad={ground_range(sol_quad):.2f} m < R_vac={ground_range(sol_vac):.2f} m",
)

## Exercise 3 — Vacuum range vs. the analytic parabola

The vacuum limit is the one place the trajectory has an elementary closed form:
the range obeys {eq}`eq-range`. That makes it the natural correctness check on
the whole pipeline: solver, event location, and `ground_range` together must
reproduce a formula we know exactly.

1. With drag off, integrate with `scipy.integrate.solve_ivp` (`DOP853`) and a
   ground-impact `event`, and compute the range at $\theta = 30°, 45°, 60°$.
2. Compare each to $R = v_0^2 \sin 2\theta / g$ from {eq}`eq-range`. (Verified
   to $\sim10^{-15}$ with the tight solver + event location; $10^{-4}$ is a safe
   ceiling.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(R_num, R_analytic, "vacuum range matches v0^2 sin2θ/g", rtol=1e-4)

## Exercise 4 — Range vs. launch angle for all three regimes

Once range is a function we can evaluate at any angle, the whole curve
$R(\theta)$ is within reach, and {eq}`eq-range` is only the *vacuum* member of
the family. Plotting all three regimes together shows what drag does to the
shape, not just to a single number.

1. Sweep the launch angle on a fine grid and plot $R(\theta)$ for vacuum,
   linear, and quadratic drag on one axes, marking each peak.
2. Note the shape: the vacuum curve is symmetric about its $45°$ peak (the
   maximum of {eq}`eq-range`); the drag curves are lower and lean left — a first
   look at the optimal-angle shift made quantitative in Exercise 7.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(theta_opt_vacuum, 45.0, "vacuum optimum is 45°", atol=0.5)

## Exercise 5 — Linear drag matches its closed form

Turning off the quadratic term decouples the two components and leaves a linear
system: the one case with drag that integrates by hand, giving the closed form
{eq}`eq-linear`. That exact solution is a second, independent check on the
integrator: now with a drag term actually switched on.

1. Integrate the Stokes case ($k = 0.5$) with `scipy.integrate.solve_ivp`
   (`DOP853`, `dense_output=True`).
2. Compare $x(t)$ and $y(t)$ to {eq}`eq-linear` at several times. (We integrate
   without the ground event so the comparison runs past the landing point, where
   the analytic solution still holds.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    [x_num, y_num],
    [x_ana, y_ana],
    "linear drag matches closed form",
    rtol=1e-5,
)

## Exercise 6 — Quadratic terminal velocity

The quadratic case has no elementary closed form for the full trajectory, but
one number survives exactly: in free fall the speed grows until drag balances
gravity, fixing the terminal velocity {eq}`eq-terminal`. It is the checkable
fingerprint of the nonlinear regime.

1. Drop an object from rest under quadratic drag ($\kappa = 0.02$) and
   integrate long enough (~60 s, `scipy.integrate.solve_ivp`) for $v_y$ to
   flatten.
2. Confirm the asymptote equals $-\sqrt{g/\kappa}$ from {eq}`eq-terminal`.
   (Verified to $\sim10^{-10}$; $10^{-3}$ covers a shorter integration.)

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    vy[-1],
    vy_term,
    "quadratic terminal velocity = -√(g/κ)",
    rtol=1e-3,
)

## Exercise 7 — Why the optimal angle drops below 45°

The payoff. In vacuum the optimum is exactly $45°$: the peak of {eq}`eq-range`.
Drag breaks that symmetry: it penalises the long, slow, high-arc trajectories
more than the flat ones, so the range-maximising angle slides below $45°$.

1. For quadratic drag at $v_0 = 30$ m/s, find the range-maximising angle on a
   fine grid (`numpy.argmax` over the swept ranges) and confirm it falls below
   the vacuum's $45°$ optimum.
2. Track the optimum across launch speeds and watch it drop further as $v_0$
   rises.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    theta_opt_quad < 45.0,
    "drag lowers the optimal launch angle",
    f"optimal θ ≈ {theta_opt_quad:.0f}° (vacuum is 45°)",
)

## Exercise 8 — The approach to terminal velocity, across drag strengths

Exercise 6 measured the terminal velocity at one drag coefficient; here we map
how the *approach* depends on $\kappa$. Drop an object from rest under quadratic
drag for several $\kappa$ and watch each $v_y(t)$ flatten toward its own
asymptote $-\sqrt{g/\kappa}$ of {eq}`eq-terminal`: stronger drag means a lower,
sooner-reached terminal speed. This is a *comparison plot*, not an animation: the
lesson is how the family of curves differ, which a still figure shows at a glance
(a single curve creeping toward a line in real time would only obscure it).

1. For each $\kappa \in \{0.02, 0.05, 0.10\}\,\mathrm{m^{-1}}$, drop from rest
   and integrate $v_y(t)$ with `scipy.integrate.solve_ivp` on a fine grid long
   enough to flatten.
2. Plot the three $v_y(t)$ together with their asymptotes $-\sqrt{g/\kappa}$.
3. The validation checks each curve's final velocity against {eq}`eq-terminal`.

In [ ]:
# (solution hidden on the public site)


### Validation 8 — every curve reaches its predicted terminal velocity

Each drop must flatten to the terminal velocity {eq}`eq-terminal` for its own
$\kappa$. The check compares all three final velocities to $-\sqrt{g/\kappa}$.

In [ ]:
validate.close(
    finals8,
    terms8,
    "each fall reaches its terminal velocity -√(g/κ)",
    rtol=1e-3,
)

## Notebook summary

- Projectile motion integrated to impact (`scipy.integrate.solve_ivp` with a ground event)
  across three drag regimes: the vacuum range matched $v_0^2\sin2\theta/g$ with a $45°$
  optimum, and the drag-free limit recovered free fall.
- Linear drag matched its closed form; the quadratic terminal velocity $\sqrt{mg/c}$; the
  optimal launch angle dropping below $45°$ as drag grows; and the approach to terminal speed.

## Outlook

- **Targeting (a shooting problem).** With drag on, find the launch angle that
  makes the projectile pass through a given target $(x_\star, y_\star)$ by
  root-finding on the miss distance: there are generally two solutions, a flat
  and a lofted shot.
- **Reynolds-number crossover.** Estimate where linear vs. quadratic drag
  dominates for a real ball, and check the per-unit-mass coefficients used here
  against $b, c$ from the size, speed, and air properties.
- **Wind.** Add a constant horizontal wind by evaluating the drag on the
  *air-relative* velocity $\mathbf v - \mathbf v_\mathrm{wind}$, and watch the
  range become asymmetric in launch direction.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()